# 01 — Delta Lake com PySpark

Este notebook demonstra o ciclo completo de operações em uma tabela **Delta Lake** usando o modelo estrela criado em `00_modelagem_dados.ipynb`:

1. Criar SparkSession com Delta Lake.
2. Escrever as 3 tabelas (`dim_type`, `dim_generation`, `fact_pokemon`) em Delta.
3. **INSERT** de novos Pokémons (Gen 7).
4. **UPDATE** — rebalancear stats de lendários.
5. **DELETE** — remover Pokémons fracos.
6. **MERGE / Upsert** — combinar correções e novos.
7. **Schema Evolution** — adicionar coluna `mega_evolution`.
8. **Time Travel** — ler versões anteriores.
9. **DESCRIBE HISTORY** — auditoria.
10. **OPTIMIZE** + **VACUUM** — manutenção.

> **Pré-requisito**: rodar primeiro `00_modelagem_dados.ipynb` para gerar `/workspace/data/gold/`.

## 1. SparkSession com Delta Lake

Usamos `configure_spark_with_delta_pip` (helper do delta-spark) que injeta o JAR Delta correto e habilita as extensões SQL.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

DELTA_DIR = "/workspace/data/delta"
GOLD_DIR  = "/workspace/data/gold"

builder = (
    SparkSession.builder
    .appName("DeltaLakePokemon")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", f"{DELTA_DIR}/warehouse")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

## 2. Escrever gold em Delta (write inicial)

`fact_pokemon` é particionada por `generation_id`.

In [ ]:
dim_type        = spark.read.parquet(f"{GOLD_DIR}/dim_type")
dim_generation  = spark.read.parquet(f"{GOLD_DIR}/dim_generation")
fact_pokemon    = spark.read.parquet(f"{GOLD_DIR}/fact_pokemon")

dim_type.write.format("delta").mode("overwrite").save(f"{DELTA_DIR}/dim_type")
dim_generation.write.format("delta").mode("overwrite").save(f"{DELTA_DIR}/dim_generation")
(fact_pokemon
    .write.format("delta")
    .mode("overwrite")
    .partitionBy("generation_id")
    .save(f"{DELTA_DIR}/fact_pokemon"))

print("Tabelas Delta criadas em:", DELTA_DIR)
spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon").show(5)

In [ ]:
# Inspecionar metadata da tabela Delta
spark.sql(f"DESCRIBE DETAIL delta.`{DELTA_DIR}/fact_pokemon`").show(truncate=False, vertical=True)

## 3. INSERT — adicionar Pokémons fictícios da Gen 7

Antes de inserir Pokémons da Gen 7, precisamos garantir que a `dim_generation` contenha a Gen 7 (caso contrário, FK inválida). Vamos inserir tanto na dimensão quanto na fato.

In [ ]:
# 3a. INSERT na dim_generation
new_gen = spark.createDataFrame(
    [(7, "Alola", "Modern")],
    schema="generation_id INT, generation_name STRING, era STRING",
)
new_gen.write.format("delta").mode("append").save(f"{DELTA_DIR}/dim_generation")
spark.read.format("delta").load(f"{DELTA_DIR}/dim_generation").orderBy("generation_id").show()

In [ ]:
# 3b. INSERT na fact_pokemon (3 starters da Gen 7)
# Buscamos os type_id corretos da dim_type para Grass, Ghost, Fire, Dark, Water, Fairy
type_lookup = {
    row["type_name"]: row["type_id"]
    for row in spark.read.format("delta").load(f"{DELTA_DIR}/dim_type").collect()
}
print("Tipos:", type_lookup)

next_id = (
    spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon")
    .agg(F.max("pokemon_id")).first()[0]
) + 1
print("Próximo pokemon_id:", next_id)

gen7 = spark.createDataFrame(
    [
        (next_id,     "Decidueye",  type_lookup["Grass"], type_lookup["Ghost"],   78,107,75,100,100,70,530, False, 7),
        (next_id + 1, "Incineroar", type_lookup["Fire"],  type_lookup["Dark"],    95,115,90,80,90,60,530,  False, 7),
        (next_id + 2, "Primarina",  type_lookup["Water"], type_lookup["Fairy"],   80,74,74,126,116,60,530, False, 7),
    ],
    schema=("pokemon_id INT, name STRING, type_1_id INT, type_2_id INT, "
            "hp INT, attack INT, defense INT, sp_atk INT, sp_def INT, speed INT, "
            "total INT, is_legendary BOOLEAN, generation_id INT"),
)
gen7.write.format("delta").mode("append").save(f"{DELTA_DIR}/fact_pokemon")

spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon")\
     .where("generation_id = 7").show()

## 4. UPDATE — rebalancear stats de Pokémons lendários

Vamos diminuir o `attack` em 10 para todos os lendários (cenário fictício de "rebalanceamento de jogo").

In [ ]:
fact_table = DeltaTable.forPath(spark, f"{DELTA_DIR}/fact_pokemon")

before = (spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon")
          .where("is_legendary = true")
          .agg(F.avg("attack").alias("avg_attack_legendary"))
          .first()[0])
print(f"Avg attack (lendários) ANTES: {before:.2f}")

fact_table.update(
    condition = F.col("is_legendary") == True,
    set       = {"attack": F.col("attack") - F.lit(10)}
)

after = (spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon")
         .where("is_legendary = true")
         .agg(F.avg("attack").alias("avg_attack_legendary"))
         .first()[0])
print(f"Avg attack (lendários) DEPOIS: {after:.2f}")

## 5. DELETE — remover Pokémons muito fracos

Vamos deletar todos com `total < 250` (Pokémons de stats muito baixas).

In [ ]:
before_count = spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon").count()
to_delete    = spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon").where("total < 250").count()
print(f"Linhas antes: {before_count} | Serão deletadas: {to_delete}")

fact_table.delete(F.col("total") < F.lit(250))

after_count = spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon").count()
print(f"Linhas depois: {after_count}")

## 6. MERGE / Upsert — corrigir stats existentes + inserir novos

Cenário: chegou uma fonte com **correções de attack** para Pokémons existentes E **novos Pokémons** que ainda não estão na tabela. Um único `MERGE` resolve as duas operações.

In [ ]:
# Fonte de updates (alguns existentes + 1 novo)
max_id = (spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon")
          .agg(F.max("pokemon_id")).first()[0])
new_id = max_id + 1

updates = spark.createDataFrame(
    [
        # Correção: ajustar attack do Bulbasaur (#1)
        (1,      "Bulbasaur", type_lookup["Grass"], type_lookup["Poison"], 45, 55, 49, 65, 65, 45, 318, False, 1),
        # Correção: ajustar attack do Charmander (#4)
        (4,      "Charmander", type_lookup["Fire"], None,                  39, 58, 43, 60, 50, 65, 309, False, 1),
        # Novo Pokémon (Gen 7) — Mimikyu
        (new_id, "Mimikyu",   type_lookup["Ghost"], type_lookup["Fairy"],  55, 90, 80, 50, 105, 96, 476, False, 7),
    ],
    schema=("pokemon_id INT, name STRING, type_1_id INT, type_2_id INT, "
            "hp INT, attack INT, defense INT, sp_atk INT, sp_def INT, speed INT, "
            "total INT, is_legendary BOOLEAN, generation_id INT"),
)
updates.show()

In [ ]:
(fact_table.alias("t")
    .merge(updates.alias("s"), "t.pokemon_id = s.pokemon_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon")\
    .where("pokemon_id in (1, 4) or name = 'Mimikyu'").show()

## 7. Schema Evolution — adicionar coluna `mega_evolution`

Delta Lake suporta **schema evolution** com `mergeSchema=true` no append (ou autoMerge habilitado globalmente como na SparkSession).

In [ ]:
# Append de uma linha que tem a coluna nova → schema evolution
mega_id = (spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon")
           .agg(F.max("pokemon_id")).first()[0]) + 1

mega = spark.createDataFrame(
    [(mega_id, "Mega Charizard X", type_lookup["Fire"], type_lookup["Dragon"],
      78, 130, 111, 130, 85, 100, 634, False, 6, True)],
    schema=("pokemon_id INT, name STRING, type_1_id INT, type_2_id INT, "
            "hp INT, attack INT, defense INT, sp_atk INT, sp_def INT, speed INT, "
            "total INT, is_legendary BOOLEAN, generation_id INT, mega_evolution BOOLEAN"),
)
(mega.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(f"{DELTA_DIR}/fact_pokemon"))

# Confirmar nova coluna
spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon").printSchema()
spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon")\
     .where("mega_evolution = true").show()

## 8. Time Travel

Cada operação cria uma nova **versão** da tabela. Podemos ler qualquer versão anterior por `versionAsOf` ou `timestampAsOf`.

In [ ]:
# Versão 0 = write inicial (antes de qualquer INSERT/UPDATE/DELETE/MERGE)
v0 = (spark.read.format("delta")
      .option("versionAsOf", 0)
      .load(f"{DELTA_DIR}/fact_pokemon"))
print(f"Versão 0 — total de linhas: {v0.count()}")

# Versão atual
current = spark.read.format("delta").load(f"{DELTA_DIR}/fact_pokemon")
print(f"Versão atual — total de linhas: {current.count()}")

# Diferença: lendários têm attack diferente entre v0 e atual?
v0_legendary_attack = v0.where("is_legendary = true").agg(F.avg("attack")).first()[0]
now_legendary_attack = current.where("is_legendary = true").agg(F.avg("attack")).first()[0]
print(f"Avg attack lendários — v0: {v0_legendary_attack:.2f} | atual: {now_legendary_attack:.2f}")

## 9. DESCRIBE HISTORY — auditoria completa

Mostra todas as operações registradas no `_delta_log/`.

In [ ]:
(spark.sql(f"DESCRIBE HISTORY delta.`{DELTA_DIR}/fact_pokemon`")
  .select("version", "timestamp", "operation", "operationMetrics")
  .show(truncate=False))

## 10. OPTIMIZE + VACUUM

- `OPTIMIZE` — compacta arquivos pequenos em maiores (melhora performance de leitura).
- `VACUUM` — remove arquivos físicos órfãos. Por padrão respeita 7 dias de retenção; configuramos `retentionDurationCheck.enabled=false` apenas para demo.

In [ ]:
spark.sql(f"OPTIMIZE delta.`{DELTA_DIR}/fact_pokemon`").show(truncate=False)

In [ ]:
# VACUUM com retenção 0 horas — APENAS PARA DEMO. Em produção, manter padrão (168h = 7 dias).
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql(f"VACUUM delta.`{DELTA_DIR}/fact_pokemon` RETAIN 0 HOURS").show(truncate=False)

In [ ]:
spark.stop()

## Conclusões

- Demonstramos o **CRUD completo** sobre uma tabela Delta + **MERGE** + **schema evolution** + **time travel** + manutenção (OPTIMIZE/VACUUM).
- Cada operação é registrada como uma **versão** no `_delta_log/`, permitindo auditoria e rollback.
- Em seguida, no notebook `02_apache_iceberg.ipynb`, repetiremos o mesmo cenário com **Apache Iceberg** e mostraremos um recurso que **só Iceberg tem**: *partition evolution*.